# Exercise: Train Your Own Image Classifier

In Lesson 2 we trained a pet breed classifier together, walking through every step. Now it's your turn to do it on a completely different dataset.

Choose a dataset below, then follow the process steps. Use Claude Code when you get stuck — the goal isn't memorizing fastai syntax, it's internalizing the process. Every ML project, from a weekend experiment to a production system, follows these same steps.

## Dataset Options

Each dataset has a different structure, different number of classes, and different quirks. That's intentional — part of the exercise is figuring out how to adapt your approach to the data you're given.

### 1. Oxford Flowers 102

**What it is:** 8,189 images of 102 flower species. High-quality real photographs.

**Download:** Available through `torchvision.datasets.Flowers102` or directly from the [Visual Geometry Group](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/). ~350 MB.

| Pros | Cons |
|------|------|
| Beautiful, high-resolution images | 102 classes is a lot |
| Real practical application (plant identification) | Labels come as integer indices, not names |
| Comes with predefined train/val/test splits | Need to map indices → species names yourself |
| Transfer learning handles fine-grained well | Some species look very similar |

**Expected accuracy with transfer learning:** ~85-95% depending on architecture and epochs

**Data structure challenge:** Labels are stored in a `.mat` file (MATLAB format), not in folder names or filenames. You'll need to figure out how to load them and connect each image to its label.

### 2. EuroSAT

**What it is:** 27,000 satellite images across 10 land-use categories (forest, river, highway, residential, industrial, etc.).

**Download:** Available through `torchvision.datasets.EuroSAT` or from [GitHub](https://github.com/phelber/EuroSAT). ~90 MB.

| Pros | Cons |
|------|------|
| Small download, fast to get started | Satellite images look very different from ImageNet |
| Only 10 classes — high accuracy is achievable | Some categories hard to distinguish from above |
| Interesting domain (remote sensing) | All images are 64×64 — small |
| Organized in folders per class — easy structure | Less visual variety than natural photos |

**Expected accuracy with transfer learning:** ~90-97%

**Data structure challenge:** Straightforward folder-per-class structure. The real question is: does a model trained on natural photos (ImageNet) transfer well to aerial imagery? You might be surprised.

### 3. Caltech-101

**What it is:** 9,146 images across 101 object categories — faces, airplanes, guitars, lobsters, etc. A classic benchmark from Caltech.

**Download:** Available through `torchvision.datasets.Caltech101` or from [Caltech](https://data.caltech.edu/records/mzrjq-6wc02). ~130 MB.

| Pros | Cons |
|------|------|
| Wide variety of objects — fun to browse | Heavily imbalanced (30–800 images per class) |
| Folder-per-class structure | Some images are grayscale, others RGB |
| Good size for experimentation | Includes a "BACKGROUND" class you may want to exclude |
| Real-world object recognition task | Image quality varies significantly |

**Expected accuracy with transfer learning:** ~85-93%

**Data structure challenge:** Class imbalance is the main issue. Some classes have 30× more images than others. You'll see this directly in the confusion matrix — does the model just ignore rare classes? What could you do about it?

### 4. DTD (Describable Textures)

**What it is:** 5,640 images of 47 texture categories — striped, dotted, woven, cracked, marbled, etc.

**Download:** Available through `torchvision.datasets.DTD` or from [Visual Geometry Group](https://www.robots.ox.ac.uk/~vgg/data/dtd/). ~600 MB.

| Pros | Cons |
|------|------|
| Tests transfer learning in a unique way | 600 MB is the largest download here |
| 47 classes — middle ground of difficulty | Some texture categories are subjective |
| Comes with predefined train/val/test splits | Smaller dataset (5.6k images total) |
| No "objects" — purely about visual patterns | Some categories overlap visually |

**Expected accuracy with transfer learning:** ~65-75%

**Data structure challenge:** Textures have no clear "object" — the entire image is the pattern. ImageNet models learned to recognize objects. Does that knowledge transfer to recognizing textures? The accuracy gap compared to other datasets here tells you something about what transfer learning actually captures.

## Recommendations

### Top pick: EuroSAT

Smallest download, clean folder structure, only 10 classes so you'll get satisfying accuracy quickly. The interesting twist is the domain shift — satellite imagery is nothing like the natural photos the model was pre-trained on. You'll see firsthand whether transfer learning generalizes across visual domains.

### For a challenge: Flowers 102

The data loading is harder (labels in a `.mat` file, not folder names), and 102 fine-grained classes pushes the model. If you want to spend more time on Step 1 figuring out the data structure, this is the one.

### For a real-world lesson: Caltech-101

The class imbalance is instructive. You'll see the model perform great on common classes and terribly on rare ones — exactly what happens with real-world data. The mixed grayscale/RGB images also require handling.

## The Process

Every supervised ML project follows the same loop:

```
1. Understand the data     →  What am I looking at? How is it organized?
2. Prepare the data        →  Get it into a format the model can consume
3. Train                   →  Let the model learn from the training data
4. Evaluate                →  How well does it work? Where does it fail?
5. Iterate                 →  Change something, re-train, compare
6. Ship                    →  Save and optionally deploy
```

The rest of this notebook walks you through each step. The code cells are empty — that's the exercise.

## Phase 1: Understand Your Data

*Goal: Know what you're working with before writing any training code.*

This is the step most beginners skip, and it's the step that causes the most problems downstream. A misunderstanding about how labels are stored, or not noticing that some images are grayscale, will break your pipeline later.

**Task 1.1: Download and inspect the file structure**

Download your chosen dataset. Before loading anything into Python, look at how the files are organized on disk.

| Structure | What it looks like | How labels work |
|-----------|-------------------|------------------|
| Folder per class | `data/forest/img1.jpg`, `data/river/img2.jpg` | Folder name = label |
| Flat folder + label file | `images/00001.jpg` + `labels.mat` or `labels.csv` | Look up label by filename or index |
| torchvision object | `dataset[0]` returns `(image, label_index)` | Integer index, need class name mapping |

Which structure does your dataset use? This determines how you'll set up the DataBlock in Phase 2.

In [ ]:
# Download the dataset and explore the file structure


**Task 1.2: Look at the images**

Load and display a handful of random images with their labels. Answer:

- Do the labels make sense for the images you see?
- Are the images high quality or noisy?
- Are they all the same size, or do dimensions vary?
- Are they all RGB, or are some grayscale?

In [ ]:
# Display several sample images with their labels


**Task 1.3: Check the class distribution**

Count how many images are in each class. Plot it.

- Is the dataset balanced (roughly equal per class) or imbalanced?
- If imbalanced: which classes have the most? The least? How extreme is the difference?
- How many total classes are there?

This matters because heavily imbalanced datasets can cause the model to ignore rare classes entirely.

In [ ]:
# Count and visualize the class distribution


## Phase 2: Prepare the Data

*Goal: Build a DataBlock and DataLoaders that fastai can train on.*

This is where your understanding from Phase 1 becomes code. The DataBlock needs to know four things:

**Task 2.1: Configure the DataBlock**

Think through each parameter:

| Parameter | Question to answer | In Lesson 2 we used |
|-----------|-------------------|---------------------|
| `blocks` | What's the input type? Output type? | `(ImageBlock, CategoryBlock)` |
| `get_items` | How do you find all the images? | `get_image_files` |
| `get_y` | How do you extract the label for each image? | `RegexLabeller` on filename |
| `splitter` | How do you split train/validation? | `RandomSplitter(valid_pct=0.2)` |
| `item_tfms` | What per-image transforms? (resizing) | `Resize(460)` |
| `batch_tfms` | What batch transforms? (augmentation) | `aug_transforms(size=224)` |

Your dataset probably differs in at least `get_items` and `get_y`. That's the core puzzle.

Some things to consider:
- If your dataset comes from torchvision, you might need to save images to disk first, or build a custom `get_items` / `get_y` that works with the torchvision object
- If labels are integers, you'll need a mapping from index to name
- If the dataset has a predefined split, use `IndexSplitter` or `FuncSplitter` instead of `RandomSplitter`
- Think about whether horizontal flips make sense for your images (flowers: yes, text: no, satellite: yes)

In [ ]:
# Build your DataBlock and create DataLoaders


**Task 2.2: Verify with `show_batch()`**

Always check before training. Look at the output and confirm:

- Images look correct (not corrupted, not weirdly stretched)
- Labels match what you see in the images
- Augmentations are sensible (call `show_batch()` twice — the images should look different each time due to random transforms)

If something is wrong here, fix it now. Training on bad data is the most common waste of time in ML.

In [ ]:
# show_batch() — run this cell twice to see augmentation differences


## Phase 3: Train

*Goal: Train a model and watch the metrics.*

**Task 3.1: Create a learner and fine-tune**

Same approach as Lesson 2:
- `vision_learner` with a pre-trained architecture
- `fine_tune()` for a few epochs

Start with `resnet34` and 3-4 epochs. You can always try bigger/longer in Phase 5.

While it trains, watch the output table:

| What to watch | Good sign | Warning sign |
|---------------|-----------|---------------|
| Training loss | Going down | Stuck or jumping around |
| Validation loss | Going down, close to training loss | Going UP while training loss goes down |
| Accuracy | Going up | Stuck well below what you'd expect |

In [ ]:
# Create learner and train


## Phase 4: Evaluate

*Goal: Understand what the model learned and where it fails.*

A single accuracy number tells you almost nothing useful. This phase is where you actually learn something.

**Task 4.1: Most confused classes**

Use `ClassificationInterpretation` to find which classes the model confuses most. For each confused pair, ask yourself:

- Do these classes actually look similar? (If yes: the model is making reasonable mistakes)
- Is one of the classes underrepresented? (If yes: the model might not have enough examples)
- Would a human struggle with these too? (If yes: your model might be close to the ceiling)

In [ ]:
# Find and display the most confused class pairs


**Task 4.2: Top losses**

Look at the images where the model was most confidently wrong. These tell you more than the successes.

Common findings:
- Genuinely ambiguous images (even you can't tell)
- Mislabeled data in the dataset itself
- Unusual angles, lighting, or crops that threw the model off
- Images that arguably belong to multiple classes

In [ ]:
# Plot top losses


**Task 4.3: Make predictions on individual images**

Pick a few images and run `learn.predict()`. Display the image alongside the top predictions and their probabilities.

- Is the model confident (90%+) or uncertain (spread across classes)?
- When it's uncertain, are the top alternatives reasonable?
- Try an image that's deliberately tricky — how does the model handle it?

In [ ]:
# Predict on individual images and show results


## Phase 5: Iterate

*Goal: Improve your results by changing one thing at a time.*

Based on what you found in Phase 4, pick **one** thing to change and re-train. Compare the results.

**Task 5.1: Choose what to change**

| What to try | When it might help |
|-------------|--------------------|
| More epochs (`fine_tune(5)` or `fine_tune(8)`) | Accuracy was still climbing at the end |
| Bigger architecture (`resnet50`) | Model seems to plateau early |
| Larger image size (`Resize(320)`, `size=256`) | Fine-grained details matter (flowers, textures) |
| Adjust augmentation | Domain-specific needs (satellite: allow 90° rotations) |
| Handle class imbalance | Some classes have far fewer images |

**Important habit:** Change one variable at a time. If you change the architecture AND the image size AND the epochs, you won't know which change helped.

In [ ]:
# Re-train with your change


**Task 5.2: Compare**

Did your change help? Compare:
- Final validation accuracy: before vs after
- The confused pairs: did the same classes stay confused, or did it shift?
- Training time: was the accuracy gain worth the extra compute?

In [ ]:
# Evaluate the improved model and compare


## Phase 6: Save and (Optionally) Deploy

*Goal: Export your model so it works independently.*

**Task 6.1: Export the model**

Use `learn.export()` to save the model as a `.pkl` file. Verify it works by loading it back and making a prediction.

**Optional stretch:** The `pet_classifier_deploy` starter project from Lesson 2 works with any fastai image classifier, not just pets. Drop your `.pkl` in the `models/` folder, run `docker-compose up`, and see your model in a browser.

In [ ]:
# Export and verify


## Reflection

Before closing this notebook:

1. **What was the hardest part?** Getting the data loaded? Understanding the results? Something else?
2. **How did your dataset differ from Oxford Pets?** What did you have to change in the DataBlock, and why?
3. **What surprised you about the model's mistakes?** Did the confusions make sense, or were some unexpected?
4. **What would you try next if you had more time?**

The process you just went through — understand, prepare, train, evaluate, iterate, ship — is the same loop used at every scale of machine learning. The tools change, the data gets bigger, the models get more complex. The loop stays the same.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>